# Notebook 18 — DINNDeep on N Pacific Chl: testing v1.4 finding's generality (Track 1 v1.7)

**Goal.** nb15 showed that DINNDeep (4-channel input, ~9.4K params) drives Eq Pacific FeT r from 0.337 → **1.000** but does NOT bring recovered Carroll-6 means closer to Carroll's published values — concluded that the 5-tracer box-model proxy is the recovery ceiling, not the network. **Question for nb18: does this generalise?** Pick the next-strongest existing baseline (N Pacific Chl, DINN baseline r=0.966 from nb11) and test whether DINNDeep also saturates here with degenerate Carroll-6 recovery.

**Three possible outcomes, all useful:**

| DINNDeep r | Carroll-6 means | Reading |
|---|---|---|
| ≈ 1.000, degenerate | Don't approach Carroll's published | v1.4 finding generalises — box-model proxy IS the universal ceiling. Strengthens the cluster + carbonate-extension case. |
| ≈ DINN baseline (~0.97) | Closer to Carroll's published | v1.4 finding was Eq-Pacific-FeT-specific. Story changes; the FeT pathology is special. |
| Intermediate (e.g. 0.99) | Partial improvement | Calibrates how much of v1.4 saturation is "real degeneracy" vs "lack of constraint." |

**Setup mirrors nb15** (DINN baseline + DINNDeep head-to-head), but on the N Pacific Chl AOI / target combination from nb11. Same architecture, same hyperparameters. Only AOI + target differ.

**Loss target:** predicted phyto biomass z-scored vs Darwin Chl_total z-scored. Predicted phyto = `state[1] + state[2]` (small + large phytoplankton from carroll6). This matches nb11's loss exactly so the DINN-baseline number is a reproducibility check.

**Builds on:** nb11 (DINN baseline N Pacific Chl r=0.966), nb15 (DINNDeep on Eq Pacific FeT, the saturation finding under test).


In [ ]:
import sys
import time
import warnings
import json
import os
from pathlib import Path

_repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
_src = _repo_root / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

warnings.filterwarnings("ignore", message="Couldn't find available_diagnostics.log")
warnings.filterwarnings("ignore", category=FutureWarning)

import matplotlib.pyplot as plt
import numpy as np
import torch

from darwindiff.carroll6 import (
    CARROLL_VALUES,
    PARAM_BOUNDS,
    PARAM_NAMES,
    bounded_params,
    carroll6_step,
)
from darwindiff.diagnostics import format_pearson, safe_pearson_r
from darwindiff.ecco_darwin_loader import (
    NORTH_PACIFIC_AOI,
    open_bin_average,
    subset_aoi,
    time_mean,
    total_chlorophyll,
)
from darwindiff.networks import DINN, DINNDeep

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}, GPU={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no'}")


## 1. Load N Pacific Chl_total + 4-channel covariates from bin_average

Same data product (1° rectified `bin_average`) as nb11 / nb15. Only AOI and target differ from nb15: NORTH_PACIFIC_AOI here, Chl_total instead of FeT.

Path is env-var driven (`DARWIN_DATA_ROOT`) so this notebook is cluster-portable; default keeps local-machine behaviour.


In [ ]:
# === Data root (env-var driven for cluster portability; default keeps local behaviour) ===
DATA_ROOT = Path(os.environ.get("DARWIN_DATA_ROOT", r"D:\\ecco_darwin_v5"))
BIN_AVG_PATH = str(DATA_ROOT / "bin_average" / "v05_ECCO-Darwin_bin_average_1x1_deg.nc")

AOI = NORTH_PACIFIC_AOI

ds_bin = open_bin_average(BIN_AVG_PATH)
npacific_clim = time_mean(subset_aoi(ds_bin, AOI))

sst = npacific_clim.SST.values
mld = npacific_clim.mldDepth.values
wind = npacific_clim.windSpeed.values
lat_1d = npacific_clim.lat.values
lat_2d = np.broadcast_to(lat_1d[:, None], sst.shape).astype(np.float64)

# Target: total chlorophyll = sum(Chl1..Chl5), summed in mg/m^3
chl_total = total_chlorophyll(npacific_clim).values

ocean_mask = (
    np.isfinite(sst)
    & np.isfinite(mld)
    & np.isfinite(wind)
    & np.isfinite(chl_total)
)
n_ocean = int(ocean_mask.sum())
print(f"N Pacific bin_average climatology: {sst.shape}, ocean cells: {n_ocean}")

for name, arr in [
    ("SST", sst),
    ("MLD", mld),
    ("wind", wind),
    ("lat", lat_2d),
    ("Chl_total", chl_total),
]:
    a = arr[ocean_mask]
    print(f"  {name:>11s}: range [{a.min():.3e}, {a.max():.3e}], mean {a.mean():.3e}, std {a.std():.3e}")


## 2. Build training tensors (SST-only + 4-channel) and z-scored target

Train two networks on the SAME data: DINN baseline (SST-only, matching nb11's setup) and DINNDeep (4-channel, matching nb15's setup). Z-scored Chl_total target shared between them.


In [ ]:
def normalize(arr, mask):
    """Z-score over ocean cells, zero outside; returns float32 array."""
    ocean = arr[mask]
    return np.where(mask, (arr - ocean.mean()) / max(ocean.std(), 1e-9), 0.0).astype(np.float32)


sst_norm = normalize(sst, ocean_mask)
mld_norm = normalize(mld, ocean_mask)
wind_norm = normalize(wind, ocean_mask)
lat_norm = normalize(lat_2d, ocean_mask)

env_1ch = torch.tensor(sst_norm, dtype=torch.float32).unsqueeze(0)
env_4ch = torch.tensor(
    np.stack([sst_norm, mld_norm, wind_norm, lat_norm], axis=0),
    dtype=torch.float32,
)

chl_clean = np.where(ocean_mask, chl_total, 1.0)
chl_target = torch.tensor(chl_clean, dtype=torch.float32)
mask_t = torch.tensor(ocean_mask, dtype=torch.bool)
H, W = env_1ch.shape[1], env_1ch.shape[2]
state0 = torch.tensor([5.0e-4, 1.0, 1.0, 0.5, 0.025]).reshape(5, 1, 1).expand(5, H, W).contiguous()

env_1ch_dev = env_1ch.to(device)
env_4ch_dev = env_4ch.to(device)
state0_dev = state0.to(device)
chl_target_dev = chl_target.to(device)
mask_dev = mask_t.to(device)
bounds_dev = PARAM_BOUNDS.to(device)

chl_ocean = chl_target_dev[mask_dev]
target_mean = chl_ocean.mean()
target_std = chl_ocean.std().clamp(min=1e-6)
target_z = (chl_target_dev - target_mean) / target_std

print(f"env_1ch (SST-only) shape: {tuple(env_1ch.shape)}")
print(f"env_4ch (SST+MLD+wind+lat) shape: {tuple(env_4ch.shape)}")
print(f"Chl_total target z-scored: ocean mean={float(target_mean):.3e}, std={float(target_std):.3e}")


## 3. Train both networks: SST-only DINN baseline + 4-channel DINNDeep

Same loss (z-scored predicted phyto vs z-scored Chl_total), same hyperparameters as nb11 / nb15 (Adam lr=5e-3, 1500 epochs, 200 forward-Euler integration steps). Two training runs, ~14 min total on RTX 5090.

Loss target: `phyto = state[1] + state[2]` (Ps + Pl from carroll6) z-scored vs Darwin Chl_total z-scored. Same predicted-quantity-vs-target as nb11.


In [ ]:
DT, N_STEPS, N_EPOCHS = 0.25, 200, 1500


def train(net, env_dev, seed: int = 0) -> dict:
    torch.manual_seed(seed)
    optimizer = torch.optim.Adam(net.parameters(), lr=5e-3)
    losses = []
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for epoch in range(N_EPOCHS):
        optimizer.zero_grad()
        params = bounded_params(net(env_dev), bounds_dev)
        state = state0_dev
        for _ in range(N_STEPS):
            state = carroll6_step(state, params, DT)
        # Predicted phyto biomass = small + large phytoplankton (matches nb11 loss target)
        phyto = state[1] + state[2]
        phyto_ocean = phyto[mask_dev]
        phyto_z = (phyto - phyto_ocean.mean()) / phyto_ocean.std().clamp(min=1e-6)
        residual = (phyto_z - target_z) * mask_dev.to(phyto.dtype)
        loss = (residual ** 2).sum() / mask_dev.sum().to(residual.dtype)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if (epoch + 1) % 250 == 0:
            print(f"    epoch {epoch+1:4d}  loss = {loss.item():.4e}")
    if device == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - t0
    with torch.no_grad():
        params_final = bounded_params(net(env_dev), bounds_dev).cpu()
        state = state0_dev
        for _ in range(N_STEPS):
            state = carroll6_step(state, bounded_params(net(env_dev), bounds_dev), DT)
        phyto_final = (state[1] + state[2]).cpu()
    return {
        "losses": losses,
        "params_final": params_final,
        "phyto_final": phyto_final,
        "elapsed": elapsed,
    }


torch.manual_seed(0)
dinn_baseline = DINN(n_input_channels=1, hidden_dim=16, n_outputs=6).to(device)
n_b = sum(p.numel() for p in dinn_baseline.parameters())
print(f"=== DINN baseline (SST-only, {n_b} params) ===")
r_baseline = train(dinn_baseline, env_1ch_dev)
print(f"  done in {r_baseline['elapsed']:.0f}s, loss {r_baseline['losses'][0]:.3e} -> {r_baseline['losses'][-1]:.3e}")

torch.manual_seed(0)
dinn_deep = DINNDeep(n_input_channels=4, hidden_dim=32, n_outputs=6, n_blocks=4).to(device)
n_d = sum(p.numel() for p in dinn_deep.parameters())
print(f"\n=== DINNDeep (SST+MLD+wind+lat, {n_d} params) ===")
r_deep = train(dinn_deep, env_4ch_dev)
print(f"  done in {r_deep['elapsed']:.0f}s, loss {r_deep['losses'][0]:.3e} -> {r_deep['losses'][-1]:.3e}")


## 4. Pearson r + recovered Carroll-6 comparison


In [ ]:
for r in [r_baseline, r_deep]:
    assert torch.isfinite(r["phyto_final"][mask_t]).all(), "phyto integration produced NaN"

n_total = int(ocean_mask.sum())
target_raw = chl_total[ocean_mask]
result_b = safe_pearson_r(r_baseline["phyto_final"].numpy()[ocean_mask], target_raw)
result_d = safe_pearson_r(r_deep["phyto_final"].numpy()[ocean_mask], target_raw)

print("Pearson correlation, predicted phyto vs Darwin Chl_total (N Pacific):")
print(f"  DINN baseline   (SST only,    {n_b:>5} params):  r = {format_pearson(result_b, n_total=n_total)}")
print(f"  DINNDeep        (4-ch input,  {n_d:>5} params):  r = {format_pearson(result_d, n_total=n_total)}")
print()
print(f"Loss plateau:")
print(f"  DINN baseline:  {r_baseline['losses'][-1]:.4f}")
print(f"  DINNDeep:       {r_deep['losses'][-1]:.4f}")
if r_baseline['losses'][-1] > 0:
    improvement = (r_baseline['losses'][-1] - r_deep['losses'][-1]) / r_baseline['losses'][-1] * 100
    print(f"  Loss reduction: {improvement:.1f}%")

print("\nRecovered Carroll-6 means:")
print(f"  {'param':<11s} {'DINN baseline mean':>20s} {'DINNDeep mean':>16s} {'Carroll published':>17s}")
for i, name in enumerate(PARAM_NAMES):
    p_b = r_baseline["params_final"][i].numpy()[ocean_mask]
    p_d = r_deep["params_final"][i].numpy()[ocean_mask]
    pub = float(CARROLL_VALUES[i])
    print(f"  {name:<11s} {p_b.mean():>20.4e} {p_d.mean():>16.4e} {pub:>17.4e}")

# How much closer (or further) are DINNDeep means to Carroll vs DINN baseline?
print("\nRelative distance to Carroll's published values:")
print(f"  {'param':<11s} {'DINN |Δ|/Carroll':>17s} {'DINNDeep |Δ|/Carroll':>22s} {'closer?':>10s}")
for i, name in enumerate(PARAM_NAMES):
    p_b = r_baseline["params_final"][i].numpy()[ocean_mask].mean()
    p_d = r_deep["params_final"][i].numpy()[ocean_mask].mean()
    pub = float(CARROLL_VALUES[i])
    rel_b = abs(p_b - pub) / abs(pub) if pub != 0 else float("inf")
    rel_d = abs(p_d - pub) / abs(pub) if pub != 0 else float("inf")
    closer = "YES" if rel_d < rel_b else "no"
    print(f"  {name:<11s} {rel_b:>17.3f} {rel_d:>22.3f} {closer:>10s}")


## 5. Plots — target Chl_total, baseline DINN prediction, DINNDeep prediction, loss curves


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
chl_plot = np.where(ocean_mask, chl_total, np.nan)
phyto_b = np.where(ocean_mask, r_baseline["phyto_final"].numpy(), np.nan)
phyto_d = np.where(ocean_mask, r_deep["phyto_final"].numpy(), np.nan)

im0 = axes[0, 0].imshow(chl_plot, origin="lower", aspect="auto", cmap="viridis")
axes[0, 0].set_title("Darwin Chl_total target (N Pacific)")
plt.colorbar(im0, ax=axes[0, 0])

im1 = axes[0, 1].imshow(phyto_b, origin="lower", aspect="auto", cmap="plasma")
axes[0, 1].set_title(f"DINN baseline (SST-only)\n(r = {result_b.r:.3f})")
plt.colorbar(im1, ax=axes[0, 1])

im2 = axes[1, 0].imshow(phyto_d, origin="lower", aspect="auto", cmap="plasma")
axes[1, 0].set_title(f"DINNDeep (4-channel)\n(r = {result_d.r:.3f})")
plt.colorbar(im2, ax=axes[1, 0])

axes[1, 1].semilogy(r_baseline["losses"], label=f"DINN baseline ({n_b} params)", color="tab:red")
axes[1, 1].semilogy(r_deep["losses"], label=f"DINNDeep ({n_d} params)", color="tab:green")
axes[1, 1].set_title("Loss curves")
axes[1, 1].set_xlabel("epoch")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


## What this tests — and where it lands in the project arc

**The decision rule**: if DINNDeep on N Pacific Chl saturates to r ≈ 1.000 with degenerate Carroll-6 recovery (i.e., "closer to Carroll" mostly says "no" in the table above), the v1.4 finding from nb15 is general. If DINNDeep instead lifts r modestly (e.g., 0.97 → 0.99) AND moves Carroll-6 means closer to published values, the finding was Eq-Pacific-FeT-specific and the project arc reorders.

**Either outcome strengthens the email to Jonathan:**

- **Generalises** → the cluster ask is for box-model carbonate extension and multi-tracer joint loss specifically. The path forward isn't more capacity, it's better physics in the differentiable substrate.
- **Doesn't generalise** → understanding *why* Eq Pacific FeT is pathological becomes its own scientific question, and the cluster ask shifts toward systematic per-AOI architecture sweeps.

**Where this fits in the project arc:**
- 09–14: SST-only DINN fits across (AOI × target) combos — established the structural-ceiling argument
- 15: DINNDeep on Eq Pacific FeT — Track 1 v1.4, "saturate but degenerate"
- 16: cross-validation on nb15 — Track 1 v1.5, "interpolation only"
- 17: ensemble disagreement — Track 1 v1.6, "tail-detector, not extrapolation flag"
- **18 (this notebook): does v1.4 generalise to N Pacific Chl?** — Track 1 v1.7
- After this: cluster transfer + box-model carbonate extension + multi-tracer joint loss + Track 2 emulator.
